Here is the code where you put in H, M, G, model and measurement

In [66]:
#| default_exp prediction_module

In [15]:
#| export
import pickle
import pandas as pd 
from anthropmass.data_module import *
from anthropmass.anthro_module import *
from anthropmass.bambi_module import *

This functions gets the pickled model

In [16]:
#| export
def get_pickled_model(kindofmodel:str, measurement:str):
    filepath = f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}'
    with open(filepath,'rb') as file:
        model=pickle.load(file)
    return model

Predict_from_model uses the pickled model and the "minus mean person" to predict the measurement. One of the arguments passed into this functions is "kind of model", and currently there are three different models to chose between. The three models are xgboost, bambi, and bambi with component as input. When using bambi with component as input, the component needs to be passed as an argument into the functions, otherwise Army Reserve is set as default. The three different components to pass into the function are Regular Army, Army National Guard, and Army Reserve.

In [22]:
#| export
def predict_from_model(pickledmodel, kindofmodel:str, measurement:str, input_person:pd.DataFrame, c=False):
    person = minus_mean(input_person,['weightkg','stature'])
    if kindofmodel=='xgboost':
        return pickledmodel.predict(person)
    
    elif kindofmodel=='bambi':
        formula = 'weightkg + stature + C(Gender)'
        model = make_model_formula(measurement, formula)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    elif kindofmodel=='bambi_c':
        if isinstance(c, pd.Series):
            person['Component']=c

        elif c!= False:
            person['Component']=c
        
        else:
            person['Component']='Army Reserve'
        formula='0 + C(Gender) + Component + weightkg + stature'
        model = make_model_formula(measurement, formula)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    else:
        return 'wrong model name'

Predict for group is a function that itterates through a dataframe with multiple people. THe argument n is used when you only want to predict a part of the dataframe. 

In [ ]:
def predict_column(kindofmodel:str, measurement:str, group:pd.DataFrame, c=False):
    pickledmodel = get_pickled_model(kindofmodel, measurement)
    preds = predict_from_model(
    pickledmodel,
    kindofmodel,
    measurement,
    group[['weightkg','stature','Gender']], c)

    return preds

This function allows us to 

In [ ]:
def loop_measurements(kindofmodel:str, measurements:list, group:pd.DataFrame, c=False):
    preds_all=pd.DataFrame()
    for m in measurements:
        preds_all[m]=predict_column(kindofmodel, m, group, c)
    return preds_all

om inte comp då välja skriva str valfri c, men om har comp lägg in df

The code below is an example of a group of individuals (test) where every measurement from ansur can be predicted (all names). 

In [20]:
test= pd.read_csv('../data/processed/ANSURIItest.csv')
all_names= measurement_names()

In [11]:
xgboost=loop_measurements('xgboost', all_names, test)
test.to_csv('../output/anthro_results/predict_all_test_xgboost.csv', index=False)

ModuleNotFoundError: No module named 'xgboost'

In [21]:
bambi=loop_measurements('bambi', all_names, test)
test.to_csv('../output/anthro_results/predict_all_test_bambi.csv', index=False)

In [13]:
bambi_c=loop_measurements('bambi_c', all_names, test, test['Component'])
test.to_csv('../output/anthro_results/predict_all_test_bambi_c.csv', index=False)

In [14]:
bambi_c_AR=loop_measurements('bambi_c', all_names, test, 'Army Reserve')
test.to_csv('../output/anthro_results/predict_all_test_bambi_c_AR.csv', index=False)